# ForgeLM — KTO Binary Feedback Alignment

Align a model using simple thumbs-up/thumbs-down feedback — no paired preferences needed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/kto_binary_feedback.ipynb)

In [ ]:
# Install ForgeLM (clean install to avoid Colab dependency conflicts)
!pip install -q --no-cache-dir git+https://github.com/cemililik/ForgeLM.git
!pip uninstall -y wandb -q  # Remove conflicting wandb (not needed, tensorboard is default)

In [ ]:
# KTO dataset format: prompt + completion + label (boolean)
import json

kto_data = [
    {"prompt": "What is Python?", "completion": "Python is a versatile programming language.", "label": True},
    {"prompt": "What is Python?", "completion": "Python is a snake.", "label": False},
    {"prompt": "Explain recursion.", "completion": "Recursion is when a function calls itself.", "label": True},
    {"prompt": "Explain recursion.", "completion": "I don't know.", "label": False},
]

with open("kto_data.jsonl", "w") as f:
    for item in kto_data:
        f.write(json.dumps(item) + "\n")

print(f"Created {len(kto_data)} KTO examples")

In [ ]:
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-1.7B-Instruct"
  max_length: 1024
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  trainer_type: "kto"       # Binary feedback alignment
  kto_beta: 0.1
  output_dir: "./kto_checkpoints"
  num_train_epochs: 1
  per_device_train_batch_size: 2
  learning_rate: 5.0e-6

data:
  dataset_name_or_path: "kto_data.jsonl"
"""

with open("kto_config.yaml", "w") as f:
    f.write(config_yaml)
print("Config saved!")

In [ ]:
!forgelm --config kto_config.yaml --dry-run

In [ ]:
!forgelm --config kto_config.yaml